# Workspace RayCluster + job client — Ray Train + MLflow

**Ray library this topic:** [Ray Train](https://docs.ray.io/en/latest/train/train.html) (`TorchTrainer`). **MLflow** is separate experiment tracking (not a Ray library).

**Same platform as Topics 1–2:** shared `ray-workshop` cluster + `job_client`.

**Different from Topic 1** (Ray Data) and **Topic 2** (Ray Core tasks): this job runs multi-GPU distributed training (`ScalingConfig(num_workers=2, use_gpu=True)` must match the 2 GPU workers).

Runs `train_fashion_mnist.py`. Tear the cluster down at the end of this topic.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from workshop_bootstrap import setup_workshop_path

REPO_ROOT = setup_workshop_path()
print(f"Repo root: {REPO_ROOT}")

## Authenticate

Same credentials as Topic 1 — OpenShift Console → **Copy login command** → Display token.


In [ ]:
from kube_authkit import AuthConfig, get_k8s_client
from codeflare_sdk import set_api_client, list_local_queues, view_clusters

OPENSHIFT_SERVER = "https://api.YOUR_CLUSTER:6443"
OPENSHIFT_TOKEN = "REPLACE_WITH_YOUR_TOKEN"

auth_config = AuthConfig(
    method="openshift",
    k8s_api_host=OPENSHIFT_SERVER.strip(),
    token=OPENSHIFT_TOKEN.strip(),
    verify_ssl=False,
)
set_api_client(get_k8s_client(config=auth_config))

NAMESPACE = "ray-workshop"
LOCAL_QUEUE = "ray-workshop-queue"

list_local_queues(NAMESPACE)

## MLflow (OpenShift AI managed)

Use the **dashboard** MLflow URL (not the internal `:8443` service). Auth must be your **user token** — Ray service-account tokens are rejected by the OpenShift AI gateway.

```sh
# UI / tracking base (append is already /mlflow in status.url)
oc get mlflow mlflow -n redhat-ods-applications -o jsonpath='{.status.url}{"\n"}'
```

Pass the **same** OpenShift Console token you used in the AuthConfig cell as `MLFLOW_TRACKING_TOKEN`.


In [ ]:
# From: oc get mlflow mlflow -n redhat-ods-applications -o jsonpath='{.status.url}{"\n"}'
MLFLOW_TRACKING_URI = "https://<rhoai-dashboard-host>/mlflow"
MLFLOW_WORKSPACE = NAMESPACE  # MLflow workspace == OpenShift project
MLFLOW_EXPERIMENT = "ray-workshop-fashion-mnist"
MLFLOW_REGISTERED_MODEL = "ray-workshop-fashion-mnist"

# Same token as AuthConfig above (Console → Copy login command)
MLFLOW_TRACKING_TOKEN = OPENSHIFT_TOKEN.strip()

print("MLFLOW_TRACKING_URI:", MLFLOW_TRACKING_URI)
print("MLFLOW_WORKSPACE:", MLFLOW_WORKSPACE)


## Create or reuse the RayCluster

Same **`ray-workshop`** cluster as Topics 1–2. `apply()` creates if missing, updates if already up.


In [ ]:
from codeflare_sdk import Cluster, ClusterConfiguration

# Shared name across Topics 1–3. apply() creates if missing, updates if present.
cluster = Cluster(
    ClusterConfiguration(
        name="ray-workshop",
        namespace=NAMESPACE,
        local_queue=LOCAL_QUEUE,
        num_workers=2,
        head_cpu_requests=5,
        head_cpu_limits=8,
        head_memory_requests=4,
        head_memory_limits=8,
        worker_cpu_requests=1,
        worker_cpu_limits=2,
        worker_extended_resource_requests={"nvidia.com/gpu": 1},
        worker_memory_requests=4,
        worker_memory_limits=6,
        write_to_file=False,
    )
)
cluster.apply()
cluster.wait_ready()
cluster.details()


## Submit Ray Train job

Same `job_client` pattern: Ray Job on the existing cluster (not a new pod image).

`runtime_env`:

| Field | Role |
|-------|------|
| `working_dir` | Repo with `train_fashion_mnist.py` |
| `pip` | Install `torch`, `torchvision`, `mlflow[kubernetes]>=3.11` into the **job** env on Ray pods |
| `env_vars` | Train settings + MLflow URI/token/workspace (user token — not Ray SA) |

**Ray Train:** `TorchTrainer` runs a DDP training loop across 2 GPU workers (FashionMNIST). Prefer a CUDA-capable Ray image from [Supported Configurations](https://access.redhat.com/articles/6856871).

**MLflow (after / during training):** log `train_*` / `test_*` metrics, `checkpoint.pt`, confusion counts, and register the PyTorch model. Open the MLflow UI when the job succeeds.


In [ ]:
import time

client = cluster.job_client

submission_id = client.submit_job(
    entrypoint="python extras/scripts/train_fashion_mnist.py",
    runtime_env={
        "working_dir": str(REPO_ROOT),
        "pip": ["torch", "torchvision", "mlflow[kubernetes]>=3.11"],
        "env_vars": {
            "NUM_EPOCHS": "3",
            "NUM_WORKERS": "2",
            "MLFLOW_TRACKING_URI": MLFLOW_TRACKING_URI,
            "MLFLOW_TRACKING_TOKEN": MLFLOW_TRACKING_TOKEN,
            "MLFLOW_WORKSPACE": MLFLOW_WORKSPACE,
            "MLFLOW_TRACKING_INSECURE_TLS": "true",
            "MLFLOW_EXPERIMENT": MLFLOW_EXPERIMENT,
            "MLFLOW_REGISTERED_MODEL": MLFLOW_REGISTERED_MODEL,
            "RAY_TRAIN_WORKER_GROUP_START_TIMEOUT_S": "120",
        },
    },
)
print(f"Submitted: {submission_id}")

terminal = {"SUCCEEDED", "FAILED", "STOPPED"}
deadline = time.time() + 1800

while time.time() < deadline:
    status = client.get_job_status(submission_id)
    print(f"Job {submission_id}: {status}")
    if str(status).split(".")[-1].upper() in terminal:
        break
    time.sleep(20)
else:
    raise TimeoutError(f"Timed out waiting for job {submission_id}")

print(client.get_job_logs(submission_id))


## Observe

1. Ray Dashboard from `view_clusters()` → **Jobs** for training progress.
2. MLflow UI → experiment **`ray-workshop-fashion-mnist`** → run metrics (`train_*` / `test_*`, `class_accuracy_*`) + artifacts (`checkpoint.pt`, `confusion_matrix.csv`) + registered model **`ray-workshop-fashion-mnist`**.


In [ ]:
view_clusters(NAMESPACE)

## Tear down

Release the shared workshop cluster after Topic 3 (end of the GPU lab path).


In [ ]:
cluster.down()
print("Cluster down.")

## Verify

1. Job reaches `SUCCEEDED`.
2. Logs show epoch `train_loss`/`train_accuracy`/`test_*`, `final test_accuracy`, an `MLflow run_id=...` line, and `Done. Ray Train FashionMNIST finished successfully.`
3. MLflow UI shows the experiment run, `checkpoint.pt` artifact, and registered model `ray-workshop-fashion-mnist`.
4. If the job stays PENDING, check that `NUM_WORKERS` matches GPU workers (workshop: 2).
